In [1]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

# Load the 3-day lagged dataset and sort by target date
df = (
    pd.read_csv("../data/boston_weather_3day.csv", parse_dates=["time"])
    .set_index("time")
    .sort_index()
)

# Separate features and target
X = df.drop(columns=["temperature_today"])
y = df["temperature_today"]

# Use 2006–2021 for training and validation
X_other = X.loc[X.index < "2022-01-01"]
y_other = y.loc[y.index < "2022-01-01"]

# Reserve 2022–2025 for final testing
X_test = X.loc[X.index >= "2022-01-01"]
y_test = y.loc[y.index >= "2022-01-01"]

print("Other:", X_other.shape, y_other.shape)
print("Test:", X_test.shape, y_test.shape)
print()

# Define 5 time-series splits
tscv = TimeSeriesSplit(n_splits=5)
cv_splits = list(tscv.split(X_other))

# Check the training and validation periods for each split
for i, (train_index, val_index) in enumerate(cv_splits):
    X_train = X_other.iloc[train_index]
    y_train = y_other.iloc[train_index]

    X_val = X_other.iloc[val_index]
    y_val = y_other.iloc[val_index]

    print("Fold", i + 1)
    print(
        "Train:",
        X_train.index.min().date(),
        "to",
        X_train.index.max().date(),
        f"({len(X_train)} rows)"
    )
    print(
        "Validation:",
        X_val.index.min().date(),
        "to",
        X_val.index.max().date(),
        f"({len(X_val)} rows)"
    )
    print()

Other: (5837, 12) (5837,)
Test: (1461, 12) (1461,)

Fold 1
Train: 2006-01-08 to 2008-09-10 (977 rows)
Validation: 2008-09-11 to 2011-05-10 (972 rows)

Fold 2
Train: 2006-01-08 to 2011-05-10 (1949 rows)
Validation: 2011-05-11 to 2014-01-06 (972 rows)

Fold 3
Train: 2006-01-08 to 2014-01-06 (2921 rows)
Validation: 2014-01-07 to 2016-09-04 (972 rows)

Fold 4
Train: 2006-01-08 to 2016-09-04 (3893 rows)
Validation: 2016-09-05 to 2019-05-04 (972 rows)

Fold 5
Train: 2006-01-08 to 2019-05-04 (4865 rows)
Validation: 2019-05-05 to 2021-12-31 (972 rows)



In [5]:
# Persistence baseline: predict the target day's temperature using the previous day's temperature

import numpy as np
from sklearn.metrics import mean_absolute_error

baseline_mae = []

for i, (_, val_index) in enumerate(cv_splits):
    y_val = y_other.iloc[val_index]

    # Previous day's temperature is the prediction
    y_pred = X_other.iloc[val_index]["temperature_2m_mean_lag1"]

    mae = mean_absolute_error(y_val, y_pred)
    baseline_mae.append(mae)

    print(f"Fold {i + 1} MAE: {mae:.3f} °C")

mean_mae = np.mean(baseline_mae)
std_mae = np.std(baseline_mae)

print(f"Mean validation MAE: {mean_mae:.3f} ± {std_mae:.3f} °C")

Fold 1 MAE: 2.361 °C
Fold 2 MAE: 2.222 °C
Fold 3 MAE: 2.493 °C
Fold 4 MAE: 2.591 °C
Fold 5 MAE: 2.353 °C
Mean validation MAE: 2.404 ± 0.127 °C


In [6]:
# Model 1: Linear Regression
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

LinReg = LinearRegression()
pipe = make_pipeline(StandardScaler(), LinReg)

grid = GridSearchCV(
    pipe,
    param_grid={},
    scoring="neg_mean_absolute_error",
    cv=tscv,
    return_train_score=True,
    verbose=0
)

grid.fit(X_other, y_other)
results = pd.DataFrame(grid.cv_results_)

# Validation MAE for each fold
for i in range(tscv.n_splits):
    mae = -results.loc[grid.best_index_, f"split{i}_test_score"]
    print(f"Fold {i + 1} MAE: {mae:.3f} °C")

# Mean and standard deviation across folds
mean_mae = -grid.best_score_
std_mae = results.loc[grid.best_index_, "std_test_score"]
print(f"Mean validation MAE: {mean_mae:.3f} ± {std_mae:.3f} °C")

Fold 1 MAE: 2.228 °C
Fold 2 MAE: 2.052 °C
Fold 3 MAE: 2.312 °C
Fold 4 MAE: 2.331 °C
Fold 5 MAE: 2.147 °C
Mean validation MAE: 2.214 ± 0.104 °C
